# PKG Geo — Phase A: edge-free geographic analysis

Everything here runs from `cust_c2c_metrics` + `cust_c2c_roles` + the MDM address
table. No network snapshots, no GPU pass.

## Run order

**Sections 1–3 are gates.** Run them, stop, and send the output back.
Section 1 raises if the identifier join fails — that is deliberate. Sections 4+
are worthless on a bad join and would produce confident-looking wrong numbers.

| | |
|---|---|
| **1** | GATE — identifier join `node` ↔ `mdm_id` |
| **2** | GATE — PKG `version`, and what `strength` / `net_flow` actually mean |
| **3** | GATE — roles table schema (never seen it; profile before designing) |
| 4 | Geo block |
| 5 | Geo units (ZIP3, labelled) |
| 6 | Coverage across three populations |
| 7 | **The Pittsburgh answer, strength-weighted** |
| 8 | **Metro net flow balance** |
| 9 | Market ranking → orphaned-dollar candidates |
| 10 | Role × geography |
| 11 | Hub and counterparty-class mix × geography |
| 12 | **Community geographic cohesion → locality skeleton** |
| 13 | Report |

## Geographic unit

ZIP3, labelled with its modal city. CBSA is the correct unit and needs the HUD USPS
ZIP→ZCTA→county crosswalk, which is an external file we do not have yet. Everything
below is written against a column called `geo_unit` so swapping in CBSA later is a
one-cell change.

In [ ]:
import json
import os
from datetime import datetime

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

In [ ]:
ADDR_TABLE = "dsihd01p_dsi.neo4j_address"
PKG_METRICS_TABLE = "bdahd01p_dlcdi1_cdi_tm.cust_c2c_metrics"
PKG_ROLES_TABLE = "bdahd01p_dlcdi1_cdi_tm.cust_c2c_roles"

WORK_DIR = None                  # e.g. "hdfs:///user/sa15474/pkg/geo" — reused if present
OUT_DIR_LOCAL = "../metrics/geo"

SNAP_COL = "load_dt"
PKG_VERSION = None               # set from section 2 output, then re-run from section 4
PKG_TIME_MIN = None
PKG_TIME_MAX = None

MIN_JOIN_PCT = 50.0              # below this, section 1 raises
MIN_UNIT_NODES = 25              # k-anonymity floor for any geo_unit shown
PIT_LAT, PIT_LON = 40.4406, -79.9959

REPORT = {"generated_at": datetime.now().isoformat(timespec="seconds"), "phase": "A"}


def note(k, v):
    REPORT[k] = v
    print(f"  {k}: {v}")


def show(df, n=40, truncate=False):
    df.show(n, truncate=truncate)


spark = (
    SparkSession.builder.appName("pkg_geo_phase_a")
    .config("spark.sql.shuffle.partitions", "800")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .config("spark.shuffle.io.maxRetries", "10")
    .enableHiveSupport()
    .getOrCreate()
)
print("Spark", spark.version)


def haversine_km(lat1, lon1, lat2, lon2):
    dlat, dlon = F.radians(lat2 - lat1), F.radians(lon2 - lon1)
    a = (F.pow(F.sin(dlat / 2), 2)
         + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.pow(F.sin(dlon / 2), 2))
    return F.lit(6371.0088) * 2 * F.asin(F.sqrt(F.least(a, F.lit(1.0))))

## 1. GATE — the identifier join

`mdm_id` is `varchar(50)`. `node` is a string. If PKG nodes are account-level or
hashed rather than party-level MDM ids, no normalisation rescues this and it becomes
a data-engineering request rather than an analytics problem.

In [ ]:
addr_raw = spark.table(ADDR_TABLE)
MAX_SNAP = addr_raw.agg(F.max(SNAP_COL)).collect()[0][0]
note("snapshot", str(MAX_SNAP))

snap_path = f"{WORK_DIR}/addr_snapshot_{MAX_SNAP}" if WORK_DIR else None
addr = None
if snap_path:
    try:
        addr = spark.read.parquet(snap_path)
        print("reusing", snap_path)
    except Exception:
        pass
if addr is None:
    addr = addr_raw.filter(F.col(SNAP_COL) == F.lit(MAX_SNAP)).select(
        "mdm_id", "mdm_address_id", "addr_loc_rec_type",
        "latitude_degrees", "longitude_degrees",
        "city", "state_or_province", "zip_cd", "addr_country", "addr_line_1",
    )
    if snap_path:
        addr.repartition(200).write.mode("overwrite").parquet(snap_path)
        addr = spark.read.parquet(snap_path)
        print("materialised", snap_path)

In [ ]:
mdm_ids = (
    addr.select(F.trim(F.col("mdm_id").cast("string")).alias("raw")).distinct()
    .persist(StorageLevel.DISK_ONLY)
)
pkg_ids = (
    spark.table(PKG_METRICS_TABLE)
    .select(F.trim(F.col("node").cast("string")).alias("raw")).distinct()
    .persist(StorageLevel.DISK_ONLY)
)
n_mdm, n_pkg = mdm_ids.count(), pkg_ids.count()
note("G1_mdm_parties", n_mdm)
note("G1_pkg_nodes", n_pkg)

In [ ]:
def id_shape(df, label):
    c = F.col("raw")
    return (
        df.select(
            F.lit(label).alias("side"),
            F.length(c).alias("id_len"),
            F.when(c.rlike(r"^\d+$"), "numeric")
             .when(c.rlike(r"^[A-Za-z0-9]+$"), "alnum").otherwise("other").alias("charset"),
        )
        .groupBy("side", "id_len", "charset").agg(F.count("*").alias("n"))
    )


shapes = id_shape(mdm_ids, "mdm_id").unionByName(id_shape(pkg_ids, "pkg_node")) \
                                    .orderBy("side", F.desc("n"))
show(shapes, 40)
REPORT["G1_id_shapes"] = [r.asDict() for r in shapes.collect()]

In [ ]:
NORMS = {
    "raw":         lambda c: c,
    "upper":       lambda c: F.upper(c),
    "strip_zeros": lambda c: F.regexp_replace(c, r"^0+", ""),
    "digits_only": lambda c: F.regexp_replace(c, r"[^0-9]", ""),
    "upper_alnum": lambda c: F.regexp_replace(F.upper(c), r"[^A-Z0-9]", ""),
    "zfill18":     lambda c: F.lpad(F.regexp_replace(c, r"^0+", ""), 18, "0"),
}

rows = []
for name, fn in NORMS.items():
    a = mdm_ids.select(fn(F.col("raw")).alias("k")).distinct()
    b = pkg_ids.select(fn(F.col("raw")).alias("k")).distinct()
    m = b.join(a, "k", "left_semi").count()
    rows.append({"norm": name, "matched": m, "pct": round(100 * m / max(n_pkg, 1), 3)})
    print(f"  {name:14s} {m:>12,}  {rows[-1]['pct']:>7.3f}%")

REPORT["G1_join"] = rows
BEST = max(rows, key=lambda r: r["matched"])
BEST_NORM, BEST_PCT = BEST["norm"], BEST["pct"]
note("G1_best_norm", BEST_NORM)
note("G1_best_pct", BEST_PCT)

In [ ]:
if BEST_PCT < MIN_JOIN_PCT:
    show(pkg_ids.limit(20), 20)
    show(mdm_ids.limit(20), 20)
    raise RuntimeError(
        f"Join gate FAILED: best match {BEST_PCT}% under any normalisation.\n"
        f"These are probably different identifier spaces (account vs party, or one side\n"
        f"hashed). Everything downstream would be silently wrong. Sample ids printed above —\n"
        f"send them with the shape table and stop here."
    )
norm_fn = NORMS[BEST_NORM]
print(f"gate passed at {BEST_PCT}% via '{BEST_NORM}'")

## 2. GATE — PKG `version`, and column semantics

`version` is the ablation ladder. Unfiltered, each node appears once per level and
every strength sum inflates. And before weighting anything by `strength` we should
confirm what it is rather than assume in+out.

In [ ]:
pkg_raw = spark.table(PKG_METRICS_TABLE)

v = (
    pkg_raw.groupBy("version")
    .agg(
        F.count("*").alias("n_rows"),
        F.countDistinct("node").alias("n_nodes"),        # nunique, not size
        F.countDistinct("time_key").alias("n_months"),
        F.min("time_key").alias("tk_min"),
        F.max("time_key").alias("tk_max"),
    )
    .orderBy("version")
)
show(v)
REPORT["G2_versions"] = [r.asDict() for r in v.collect()]

In [ ]:
# Column semantics. Cheap, and it prevents a whole class of quiet error.
sem = (
    pkg_raw.limit(2_000_000)
    .select(
        F.max(F.abs(F.col("strength") - (F.col("in_strength") + F.col("out_strength")))).alias("max_dev_strength_eq_in_plus_out"),
        F.max(F.abs(F.col("net_flow") - (F.col("in_strength") - F.col("out_strength")))).alias("max_dev_netflow_eq_in_minus_out"),
        F.max(F.abs(F.col("degree") - (F.col("in_degree") + F.col("out_degree")))).alias("max_dev_degree_eq_in_plus_out"),
        F.corr("degree", "n_neighbors").alias("corr_degree_nneighbors"),
    )
)
show(sem, 1)
REPORT["G2_semantics"] = sem.collect()[0].asDict()

**Reading section 2.** Near-zero deviations confirm the identities. If
`strength = in + out`, then summing strength across nodes counts every dollar twice —
fine as a consistent coverage weight, wrong if quoted as book volume. Section 8's net
flow work depends on `net_flow = in − out` holding.

**Set `PKG_VERSION` in cell 0 now, then continue.**

In [ ]:
if PKG_VERSION is None:
    raise RuntimeError("Set PKG_VERSION in cell 0 from the version table above.")

pkg = pkg_raw.filter(F.col("version") == F.lit(PKG_VERSION))
if PKG_TIME_MIN:
    pkg = pkg.filter(F.col("time_key") >= F.lit(PKG_TIME_MIN))
if PKG_TIME_MAX:
    pkg = pkg.filter(F.col("time_key") <= F.lit(PKG_TIME_MAX))

LATEST_TK = pkg.agg(F.max("time_key")).collect()[0][0]
note("G2_latest_time_key", str(LATEST_TK))

# Graph closure check: if every dollar leaves one node and arrives at another,
# net_flow sums to ~0 across the whole graph. A large residual means counterparties
# sit outside the node set, and metro net-flow figures in section 8 are then
# "net position vs the observed graph", not vs the world.
closure = pkg.filter(F.col("time_key") == F.lit(LATEST_TK)).agg(
    F.sum("net_flow").alias("sum_net_flow"),
    F.sum("strength").alias("sum_strength"),
).collect()[0].asDict()
closure["residual_pct"] = round(100 * abs(closure["sum_net_flow"] or 0) / max(closure["sum_strength"] or 1, 1), 4)
print(json.dumps(closure, indent=2, default=str))
REPORT["G2_closure"] = closure

## 3. GATE — roles table

Never seen this schema. `ga_role` is already in the metrics table, so the roles table
may be redundant, or may carry the fuller labelled taxonomy and transition history.
Profile it; do not design around it yet.

In [ ]:
try:
    roles = spark.table(PKG_ROLES_TABLE)
    roles.printSchema()
    note("G3_roles_columns", roles.columns)
    show(roles.limit(10), 10, truncate=30)
    role_cols = [c for c in roles.columns if "role" in c.lower()]
    for c in role_cols[:3]:
        show(roles.groupBy(c).agg(F.count("*").alias("n")).orderBy(F.desc("n")).limit(30), 30)
    REPORT["G3_roles_available"] = True
except Exception as e:
    print("roles table not readable:", str(e).splitlines()[0][:200])
    REPORT["G3_roles_available"] = False

## 4. Geo block

In [ ]:
NUM_RE = r"^\s*-?\d+(\.\d+)?\s*$"
lat_s = F.trim(F.coalesce(F.col("latitude_degrees"), F.lit("")))
lon_s = F.trim(F.coalesce(F.col("longitude_degrees"), F.lit("")))
lat_num = F.when(lat_s.rlike(NUM_RE), lat_s.cast("double"))
lon_num = F.when(lon_s.rlike(NUM_RE), lon_s.cast("double"))
has_geo = (
    lat_num.isNotNull() & lon_num.isNotNull()
    & ~((lat_num == 0) & (lon_num == 0))
    & lat_num.between(-90, 90) & lon_num.between(-180, 180)
).cast("int")

rec_u = F.upper(F.trim(F.coalesce(F.col("addr_loc_rec_type"), F.lit(""))))
ctry_u = F.upper(F.trim(F.coalesce(F.col("addr_country"), F.lit(""))))
zip_digits = F.regexp_replace(F.coalesce(F.col("zip_cd"), F.lit("")), r"[^0-9]", "")
is_us = F.when(ctry_u.isin("US", "USA", ""), 1).otherwise(0)
po_box = F.when(
    F.upper(F.trim(F.coalesce(F.col("addr_line_1"), F.lit("")))).rlike(r"(^| )P ?\.? ?O ?\.? ?BOX"), 1
).otherwise(0)

geo = (
    addr.select(
        norm_fn(F.trim(F.col("mdm_id").cast("string"))).alias("join_key"),
        lat_num.alias("lat"),
        lon_num.alias("lon"),
        has_geo.alias("has_geo"),
        is_us.alias("is_us"),
        rec_u.alias("rec_type"),
        po_box.alias("po_box"),
        F.when(F.length(zip_digits) >= 5, F.substring(zip_digits, 1, 5)).alias("zip5"),
        F.substring(zip_digits, 1, 3).alias("zip3"),
        F.upper(F.trim(F.coalesce(F.col("state_or_province"), F.lit("")))).alias("state"),
        F.upper(F.trim(F.coalesce(F.col("city"), F.lit("")))).alias("city"),
    )
    .withColumn(
        "geo_status",
        F.when((F.col("has_geo") == 0) & (F.col("is_us") == 0), "non_us")
         .when(F.col("has_geo") == 0, "missing")
         .when(F.col("rec_type").isin("POSTOFFICEBOX", "GENERALDELIVERY"), "placeholder")
         .when(F.col("po_box") == 1, "placeholder")
         .otherwise("valid"),
    )
    .persist(StorageLevel.DISK_ONLY)
)
note("geo_rows", geo.count())

## 5. Geo units — ZIP3, labelled

Raw ZIP3 codes are meaningless to a stakeholder, so each carries its modal city.
`km_from_pit` is computed from the ZIP3's own median coordinate, which means it also
works for parties whose individual geocode is missing.

In [ ]:
zip3_pt = (
    geo.filter((F.col("has_geo") == 1) & (F.col("is_us") == 1) & F.col("zip3").rlike(r"^\d{3}$"))
    .groupBy("zip3")
    .agg(
        F.expr("percentile_approx(lat, 0.5)").alias("u_lat"),
        F.expr("percentile_approx(lon, 0.5)").alias("u_lon"),
        F.count("*").alias("n_ref"),
    )
    .filter(F.col("n_ref") >= 20)
)

zip3_label = (
    geo.filter(F.col("zip3").rlike(r"^\d{3}$"))
    .groupBy("zip3", "city", "state").agg(F.count("*").alias("n"))
    .withColumn("rk", F.row_number().over(Window.partitionBy("zip3").orderBy(F.desc("n"))))
    .filter(F.col("rk") == 1)
    .select("zip3", F.col("city").alias("u_city"), F.col("state").alias("u_state"))
)

geo_units = (
    zip3_pt.join(zip3_label, "zip3", "left")
    .withColumn("km_from_pit", haversine_km(F.col("u_lat"), F.col("u_lon"),
                                            F.lit(PIT_LAT), F.lit(PIT_LON)))
    .withColumn("geo_unit", F.concat_ws(" / ", F.col("zip3"), F.col("u_city"), F.col("u_state")))
    .persist(StorageLevel.DISK_ONLY)
)
note("n_geo_units", geo_units.count())
show(geo_units.orderBy("km_from_pit").limit(20), 20, truncate=36)

## 6. Node frame + coverage across three populations

In [ ]:
ATTR = [c for c in [
    "degree", "n_neighbors", "strength", "in_strength", "out_strength", "net_flow",
    "flow_ratio", "hhi_in", "hhi_out", "pagerank_logw", "hits_hub_w", "hits_auth_w",
    "core_number", "betweenness_approx", "trophic_level", "clustering_coef",
    "reciprocity_amount_share", "community_id", "participation_coef", "within_module_z",
    "ga_role", "naics2", "naics_desc", "naics_known", "months_active",
    "months_since_first_seen", "hub_in_share", "hub_out_share",
    "share_in_amt_individual", "share_in_amt_biz_valid", "share_in_amt_hub",
    "share_out_amt_individual", "share_out_amt_biz_valid", "share_out_amt_hub",
    "n_payer_new", "n_payer_lost", "n_payee_new", "n_payee_lost",
    "new_payer_amount_share", "lost_payer_amount_share", "cust_name",
] if c in pkg.columns]

nodes = (
    pkg.groupBy("node")
    .agg(
        F.sum("strength").alias("strength_sum"),
        F.sum("net_flow").alias("net_flow_sum"),
        F.countDistinct("time_key").alias("n_months_present"),
    )
    .join(pkg.filter(F.col("time_key") == F.lit(LATEST_TK)).select("node", *ATTR), "node", "left")
    .withColumn("join_key", norm_fn(F.trim(F.col("node").cast("string"))))
    .join(geo.drop("city"), "join_key", "left")
    .join(geo_units.select("zip3", "geo_unit", "km_from_pit", "u_state"), "zip3", "left")
    .persist(StorageLevel.DISK_ONLY)
)
note("nodes", nodes.count())

In [ ]:
cov = nodes.agg(
    F.count("*").alias("pkg_nodes"),
    F.sum(F.when(F.col("geo_status").isNotNull(), 1).otherwise(0)).alias("matched_mdm"),
    F.sum(F.when(F.col("geo_status") == "valid", 1).otherwise(0)).alias("geo_valid"),
    F.sum("strength_sum").alias("strength_total"),
    F.sum(F.when(F.col("geo_status") == "valid", F.col("strength_sum")).otherwise(0.0)).alias("strength_valid"),
).collect()[0].asDict()
cov["pct_nodes_matched"] = round(100 * cov["matched_mdm"] / max(cov["pkg_nodes"], 1), 2)
cov["pct_nodes_valid"] = round(100 * cov["geo_valid"] / max(cov["pkg_nodes"], 1), 2)
cov["pct_strength_valid"] = round(100 * cov["strength_valid"] / max(cov["strength_total"], 1), 2)
print(json.dumps(cov, indent=2, default=str))
REPORT["coverage"] = cov

show(
    nodes.groupBy(F.coalesce(F.col("geo_status"), F.lit("not_in_mdm")).alias("geo_status"))
    .agg(
        F.count("*").alias("n_nodes"),
        F.round(F.sum("strength_sum") / 1e6, 1).alias("strength_musd"),
        F.round(F.expr("percentile_approx(degree, 0.5)"), 1).alias("med_degree"),
        F.round(F.expr("percentile_approx(strength, 0.5)"), 1).alias("med_strength"),
        F.round(F.expr("percentile_approx(clustering_coef, 0.5)"), 4).alias("med_cc"),
        F.round(F.avg("months_active"), 1).alias("avg_months_active"),
    ).orderBy(F.desc("n_nodes"))
)

**This table is the missingness-informative test (M2).** If geo-missing nodes look
structurally like geo-valid ones, the gap is ignorable. If they are systematically
smaller, newer, or less clustered, it is not.

## 7. The Pittsburgh answer, strength-weighted

At `MDM_ALL` level PA is 13.3% of parties and the book is footprint-shaped. That was
30M mostly-retail parties. This asks the same question of PKG-visible nodes weighted
by dollars — the version worth briefing.

In [ ]:
G = nodes.filter(F.col("geo_status") == "valid")

ring = (
    F.when(F.col("km_from_pit").isNull(), "9_unplaced")
    .when(F.col("km_from_pit") < 50, "0_under_50km")
    .when(F.col("km_from_pit") < 150, "1_50_150km")
    .when(F.col("km_from_pit") < 400, "2_150_400km")
    .when(F.col("km_from_pit") < 1000, "3_400_1000km")
    .otherwise("4_over_1000km")
)

wtot = Window.partitionBy()
rings = (
    G.withColumn("ring", ring).groupBy("ring")
    .agg(F.count("*").alias("n_nodes"), F.sum("strength_sum").alias("_s"))
    .withColumn("pct_of_nodes", F.round(100 * F.col("n_nodes") / F.sum("n_nodes").over(wtot), 2))
    .withColumn("pct_of_strength", F.round(100 * F.col("_s") / F.sum("_s").over(wtot), 2))
    .withColumn("strength_musd", F.round(F.col("_s") / 1e6, 1)).drop("_s")
    .orderBy("ring")
)
show(rings)
REPORT["rings"] = [r.asDict() for r in rings.collect()]

In [ ]:
states = (
    G.groupBy("state")
    .agg(F.count("*").alias("n_nodes"), F.sum("strength_sum").alias("_s"))
    .withColumn("pct_of_nodes", F.round(100 * F.col("n_nodes") / F.sum("n_nodes").over(wtot), 2))
    .withColumn("pct_of_strength", F.round(100 * F.col("_s") / F.sum("_s").over(wtot), 2))
    .withColumn("strength_musd", F.round(F.col("_s") / 1e6, 1)).drop("_s")
    .orderBy(F.desc("pct_of_strength"))
)
show(states, 40)
REPORT["states"] = [r.asDict() for r in states.limit(40).collect()]

**Read the gap between `pct_of_nodes` and `pct_of_strength`.** Where strength share
far exceeds node share, a few large customers carry that geography — concentrated and
fragile. Where node share exceeds strength share, it is a long tail of small
relationships. Same map, opposite management implications.

## 8. Metro net flow balance

`net_flow` is a node attribute, so summing it over a metro's nodes gives that metro's
net position against the rest of the observed graph — no edges needed. Novel exposure
metric out of a column that already exists.

Scope it by the closure residual from section 2: if that is large, this is net
position *versus the observed graph*, not versus the world.

In [ ]:
flows = (
    G.groupBy("geo_unit", "u_state", "km_from_pit")
    .agg(
        F.count("*").alias("n_nodes"),
        F.sum("net_flow_sum").alias("_net"),
        F.sum("strength_sum").alias("_s"),
        F.sum("in_strength").alias("_in"),
        F.sum("out_strength").alias("_out"),
    )
    .filter(F.col("n_nodes") >= MIN_UNIT_NODES)
    .withColumn("net_musd", F.round(F.col("_net") / 1e6, 1))
    .withColumn("strength_musd", F.round(F.col("_s") / 1e6, 1))
    .withColumn("net_ratio", F.round(F.col("_net") / F.col("_s"), 4))
    .drop("_net", "_s", "_in", "_out")
    .persist(StorageLevel.DISK_ONLY)
)
print("--- net SINKS (money flows in) ---")
show(flows.orderBy(F.desc("net_musd")).limit(25), 25, truncate=36)
print("--- net SOURCES (money flows out) ---")
show(flows.orderBy("net_musd").limit(25), 25, truncate=36)
REPORT["flow_sinks"] = [r.asDict() for r in flows.orderBy(F.desc("net_musd")).limit(15).collect()]
REPORT["flow_sources"] = [r.asDict() for r in flows.orderBy("net_musd").limit(15).collect()]

## 9. Market ranking → orphaned-dollar candidates

The real version of O2 needs the RM coverage territory file, which we do not have.
This is the input to it: markets ranked by dollars, flagged by distance from the
traditional footprint. Eyeball the out-of-footprint entries with material strength —
those are the shortlist.

In [ ]:
FOOTPRINT_STATES = ["PA", "OH", "NJ", "MD", "DE", "IN", "KY", "IL", "MI", "NC",
                    "VA", "WV", "DC", "GA", "AL", "FL", "SC", "MO", "WI", "TX"]

markets = (
    G.groupBy("geo_unit", "u_state", "km_from_pit")
    .agg(
        F.count("*").alias("n_nodes"),
        F.sum("strength_sum").alias("_s"),
        F.round(F.avg("months_active"), 1).alias("avg_months_active"),
        F.round(F.avg("hits_auth_w"), 6).alias("avg_auth"),
    )
    .filter(F.col("n_nodes") >= MIN_UNIT_NODES)
    .withColumn("strength_musd", F.round(F.col("_s") / 1e6, 1))
    .withColumn("pct_of_strength", F.round(100 * F.col("_s") / F.sum("_s").over(wtot), 3))
    .withColumn("in_footprint", F.when(F.col("u_state").isin(FOOTPRINT_STATES), 1).otherwise(0))
    .drop("_s")
)
print("--- top markets OUTSIDE the traditional footprint ---")
show(markets.filter(F.col("in_footprint") == 0).orderBy(F.desc("strength_musd")).limit(30), 30, truncate=36)
REPORT["out_of_footprint_markets"] = [
    r.asDict() for r in markets.filter(F.col("in_footprint") == 0)
    .orderBy(F.desc("strength_musd")).limit(25).collect()
]

## 10. Role × geography

Lift against the national baseline, not raw counts — raw counts just reproduce
market size.

In [ ]:
if "ga_role" in nodes.columns:
    base = (
        G.groupBy("ga_role").agg(F.sum("strength_sum").alias("_rs"))
        .withColumn("national_share", F.col("_rs") / F.sum("_rs").over(wtot))
        .select("ga_role", "national_share")
    )
    rg = (
        G.groupBy("geo_unit", "ga_role").agg(
            F.count("*").alias("n_nodes"), F.sum("strength_sum").alias("_s"))
        .withColumn("local_share", F.col("_s") / F.sum("_s").over(Window.partitionBy("geo_unit")))
        .withColumn("unit_nodes", F.sum("n_nodes").over(Window.partitionBy("geo_unit")))
        .filter(F.col("unit_nodes") >= MIN_UNIT_NODES)
        .join(F.broadcast(base), "ga_role", "left")
        .withColumn("lift", F.round(F.col("local_share") / F.col("national_share"), 2))
        .withColumn("strength_musd", F.round(F.col("_s") / 1e6, 1))
        .drop("_s")
    )
    show(rg.filter(F.col("strength_musd") > 1).orderBy(F.desc("lift")).limit(30), 30, truncate=36)
    REPORT["role_geo_lift"] = [
        r.asDict() for r in rg.filter(F.col("strength_musd") > 1).orderBy(F.desc("lift")).limit(25).collect()
    ]

## 11. Hub and counterparty-class mix × geography

Strength-weighted means. `share_in_amt_hub` high in a market means those customers'
inflows route through aggregators — thin visibility of the true payer, and an AML
consideration as much as a coverage one.

In [ ]:
mix_cols = [c for c in ["hub_in_share", "hub_out_share",
                        "share_in_amt_individual", "share_in_amt_biz_valid", "share_in_amt_hub",
                        "share_out_amt_individual", "share_out_amt_biz_valid", "share_out_amt_hub",
                        "clustering_coef", "trophic_level", "reciprocity_amount_share"]
            if c in nodes.columns]

wavg = lambda c: F.round(F.sum(F.col(c) * F.col("strength_sum")) / F.sum("strength_sum"), 4)

mix = (
    G.groupBy("geo_unit", "u_state")
    .agg(F.count("*").alias("n_nodes"),
         F.round(F.sum("strength_sum") / 1e6, 1).alias("strength_musd"),
         *[wavg(c).alias(f"w_{c}") for c in mix_cols])
    .filter(F.col("n_nodes") >= MIN_UNIT_NODES)
    .orderBy(F.desc("strength_musd"))
)
show(mix.limit(30), 30, truncate=30)
REPORT["geo_mix"] = [r.asDict() for r in mix.limit(25).collect()]

## 12. Community geographic cohesion → locality skeleton

The one edge-derived *grouping* already persisted at node level. Communities are
dense edge subgraphs, so a community's spatial spread proxies its members'
counterparty spread — which is what the true locality index will measure once the
edge pass runs.

**This is a proxy and community-level.** Every member of a community inherits the
same radius. `dist_to_comm_centroid_km` is node-specific and adds back some
resolution: a node at the centre of a tight community is differently placed from one
at the edge of a national one. Replace with true counterparty dispersion later, and
compare — that comparison tells you how good the proxy was.

In [ ]:
if "community_id" in nodes.columns:
    C = G.filter(F.col("community_id").isNotNull() & (F.col("strength_sum") > 0))

    cc = (
        C.groupBy("community_id")
        .agg(
            F.count("*").alias("comm_n_nodes"),
            F.sum("strength_sum").alias("comm_strength"),
            (F.sum(F.col("lat") * F.col("strength_sum")) / F.sum("strength_sum")).alias("c_lat"),
            (F.sum(F.col("lon") * F.col("strength_sum")) / F.sum("strength_sum")).alias("c_lon"),
        )
        .filter(F.col("comm_n_nodes") >= 5)
    )

    cd = (
        C.join(cc, "community_id")
        .withColumn("d_km", haversine_km(F.col("lat"), F.col("lon"),
                                         F.col("c_lat"), F.col("c_lon")))
        .persist(StorageLevel.DISK_ONLY)
    )

    # radius of gyration: sqrt( sum(w*d^2) / sum(w) )
    crad = (
        cd.groupBy("community_id", "comm_n_nodes", "comm_strength")
        .agg(F.sqrt(F.sum(F.col("strength_sum") * F.pow(F.col("d_km"), 2)) / F.sum("strength_sum"))
             .alias("comm_radius_km"),
             F.expr("percentile_approx(d_km, 0.5)").alias("comm_median_d_km"))
    )

    locality = (
        F.when(F.col("comm_radius_km") < 50, "LOCAL")
        .when(F.col("comm_radius_km") < 250, "REGIONAL")
        .when(F.col("comm_radius_km") < 1000, "MULTI_MARKET")
        .otherwise("NATIONAL")
    )
    crad = crad.withColumn("locality_proxy", locality).persist(StorageLevel.DISK_ONLY)

    show(
        crad.groupBy("locality_proxy").agg(
            F.count("*").alias("n_communities"),
            F.sum("comm_n_nodes").alias("n_nodes"),
            F.round(F.sum("comm_strength") / 1e6, 1).alias("strength_musd"),
            F.round(F.avg("comm_radius_km"), 1).alias("avg_radius_km"),
        ).orderBy("locality_proxy")
    )
    REPORT["locality_proxy"] = [r.asDict() for r in crad.groupBy("locality_proxy").count().collect()]

    show(crad.orderBy(F.desc("comm_strength")).limit(25), 25)

    node_loc = (
        cd.join(crad.select("community_id", "comm_radius_km", "locality_proxy"), "community_id")
        .select("node", "geo_unit", "u_state", "community_id", "ga_role",
                "strength_sum", "clustering_coef", "hits_auth_w",
                F.round("d_km", 1).alias("dist_to_comm_centroid_km"),
                F.round("comm_radius_km", 1).alias("comm_radius_km"), "locality_proxy")
        .persist(StorageLevel.DISK_ONLY)
    )

    # The CC x geography quadrants, proxy version. High CC + tight = real local supply
    # chain. High CC + spread = closed long-range circle, an AML-adjacent signal.
    show(
        node_loc.withColumn("cc_hi", F.when(F.col("clustering_coef") >
                            F.expr("percentile_approx(clustering_coef, 0.5)").over(wtot), "hi_cc")
                            .otherwise("lo_cc"))
        .groupBy("cc_hi", "locality_proxy")
        .agg(F.count("*").alias("n_nodes"),
             F.round(F.sum("strength_sum") / 1e6, 1).alias("strength_musd"))
        .orderBy("cc_hi", "locality_proxy"), 20
    )

    if WORK_DIR:
        node_loc.write.mode("overwrite").parquet(f"{WORK_DIR}/pkg_geo_locality_proxy_{LATEST_TK}")

## 13. Report

In [ ]:
os.makedirs(OUT_DIR_LOCAL, exist_ok=True)
if WORK_DIR:
    nodes.write.mode("overwrite").parquet(f"{WORK_DIR}/pkg_geo_nodes_{LATEST_TK}")
    flows.write.mode("overwrite").parquet(f"{WORK_DIR}/pkg_geo_unit_flows_{LATEST_TK}")

print(json.dumps(REPORT, indent=2, default=str))
with open(f"{OUT_DIR_LOCAL}/phase_a_report_{LATEST_TK}.json", "w") as fh:
    json.dump(REPORT, fh, indent=2, default=str)
print("report written")

---

## Send back

**After sections 1–3, before running anything else:**

1. `G1_join` and `G1_id_shapes` — the gate.
2. `G2_versions` — so `PKG_VERSION` gets set rather than guessed.
3. `G2_semantics` and `G2_closure` — whether `strength = in+out`, `net_flow = in−out`,
   and whether the graph is closed. Section 8 rests on the last one.
4. The roles `printSchema` — I have never seen it and cannot design around it blind.

**After the rest:** `rings`, `states`, the two flow tables, and `locality_proxy`.

## What this does not do, and why

No locality index, corridor map, gravity residual, second-order exposure, triangle
geography, or in-geo/out-geo split. All of those need per-edge distance, which needs
the snapshots. The community-radius proxy in section 12 is the stand-in, and its
purpose is partly to make the eventual edge pass a validation exercise rather than a
leap.

CBSA is the right unit and ZIP3 is the stand-in. Sourcing the HUD USPS crosswalk is a
small, separate task worth doing before any of this reaches a stakeholder — ZIP3
splits some metros and merges others, which is fine for internal reading and not fine
for a deck.